# 05c — Control for 05: stock head, re-initialised

**Why this run exists.** The P2 model (05) scored *below* the baseline on ball (0.045 vs 0.110 test mAP50-95). Two explanations are confounded:

- (a) a stride-4 head does not help here, or
- (b) building the P2 head discards pretrained weights — only layers 0–16 (297/593 tensors) transfer from `yolo11s.pt` — and 226 training images cannot relearn them in 100 epochs. The 05 learning curve supports (b) in part: val mAP50-95 0.076 at epoch 10 vs 0.400 for the baseline.

This control keeps the **stock** YOLO11s architecture but re-initialises exactly the layers 05 could not inherit (17–23: the bottom-up path and Detect). Everything else matches the baseline.

- If 05c ≈ 05, the P2 deficit is lost pretraining, not the P2 idea: the comparison 05 vs 05c isolates the head.
- If 05c ≈ 02, the P2 architecture itself hurts.

In [ ]:
SMOKE = False  # True: 1 epoch on 25% of train, written to <run_name>_smoke

In [ ]:
import sys
sys.path.insert(0, "/content/ballhawk")
import pandas as pd
import torch
import ballhawk_common as bc
from ultralytics import YOLO

In [ ]:
model = YOLO("yolo11s.pt")
layers = model.model.model
before = {n: p.detach().clone() for n, p in layers[17:].named_parameters()}
torch.manual_seed(0)
for layer in layers[17:]:
    for name, m in layer.named_modules():
        # DFL is a fixed, frozen 0..15 projection that decodes box distances, not a learned layer:
        # re-initialising it breaks box decoding for the whole run (first 05c attempt: mAP 0.000).
        if m is not layer and "dfl" not in name and hasattr(m, "reset_parameters"):
            m.reset_parameters()
layers[-1].bias_init()  # Detect prior biases, as a freshly built model gets them
dfl = layers[-1].dfl.conv.weight.detach().flatten()
assert torch.equal(dfl, torch.arange(len(dfl), dtype=dfl.dtype)), "DFL projection must stay 0..reg_max-1"
learned = {n for n in before if ".dfl." not in n}
changed = sum(not torch.equal(before[n], p) for n, p in layers[17:].named_parameters() if n in learned)
print(f"re-initialised {changed}/{len(learned)} learned parameter tensors in layers 17-23; DFL intact; layers 0-16 untouched")
assert changed > 0.9 * len(learned)

In [ ]:
record = bc.run_experiment(
    run_name="ctrl_reinit17_yolo11s_640", notebook="05c",
    variable_changed="layers 17-23 re-initialised (control for 05)",
    model_cfg="yolo11s.pt, layers 17-23 re-initialised", model=model, imgsz=640,
    notes="Control for 05: same architecture as 02, same layers lost as in 05. ultralytics 8.4.163.", smoke=SMOKE)

In [ ]:
pd.DataFrame(record["per_class"]).set_index("class").round(3)